# 02 · Yes/No probabilities on Colab

For each model: one forward pass per item, reading the next-token probabilities of "Yes" and "No" (summed over case and leading-space forms). No text is generated, so there is nothing to parse and each item takes a fraction of a second on a GPU.

1. **Runtime → Change runtime type → GPU** (T4 is enough; 7B models load in 4-bit).
2. Gated models (Llama, Gemma) need a Hugging Face token with access; the default list below uses ungated models only.
3. Run once with `SMOKE_TEST = True` (24 items per model) and look at the diagnostics, then set it to `False`.
4. Results go to Google Drive, so they survive a disconnect. Re-running resumes where it stopped. Copy the `logprobs-*.jsonl` files to `data/interim/` in the local repo for notebook 03.
5. The **CausalCoT** section at the end runs the same items with step-by-step reasoning first. If the direct run is already complete, set `RUN_DIRECT = False` and run all cells.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/laudima/causal-evaluation.git"

if IN_COLAB:
    ROOT = Path("/content/causal-evaluation")
    if not ROOT.exists():
        !git clone -q {REPO_URL} {ROOT}
    else:
        !git -C {ROOT} fetch -q origin && git -C {ROOT} reset -q --hard origin/main
    !pip install -q -e "{ROOT}[colab]"
    !git -C {ROOT} log --oneline -1
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import inspect

import causal_evaluation
from causal_evaluation.models import logprob

print(f"causal_evaluation {causal_evaluation.__version__} from {logprob.__file__}")
if "max_new_tokens" not in inspect.signature(logprob.done_item_ids).parameters:
    raise RuntimeError("Old code is loaded in memory: Runtime -> Restart session, then Run all.")

In [ ]:
# Optional Hugging Face token from Colab Secrets (key icon, name HF_TOKEN, notebook access on).
# The models are public; a token only avoids anonymous download rate limits.
if IN_COLAB:
    from google.colab import userdata

    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        print("HF_TOKEN loaded from Colab Secrets")
    except Exception:
        print("No HF_TOKEN secret found; public models still download")

In [ ]:
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/causal-evaluation/results")
else:
    OUT_DIR = Path(os.environ.get("CE_INTERIM", ROOT / "data" / "interim"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(OUT_DIR)

## Rebuild the sample and check it matches notebook 01

In [ ]:
from causal_evaluation.data.variants import build_variant_sample, write_jsonl

SEED = 20260903
SIZE_PER_VARIANT = 1000
EXPECTED_SHA = "28feb2479c3b6a59e16c9754d6b5e3752c585004da220198cf3cdc6c0ea656d2"  # variants-sample.manifest.json

items = build_variant_sample(ROOT / "data" / "raw", SIZE_PER_VARIANT, SEED)
sample_sha = write_jsonl(items, OUT_DIR / "variants-sample.jsonl")
print(f"{len(items):,} items, SHA-256 {sample_sha}")
if EXPECTED_SHA:
    assert sample_sha == EXPECTED_SHA, "Sample differs from the local one"

## Models

In [ ]:
# from huggingface_hub import login; login()  # only for gated models

# dtype: float32 keeps P(Yes) independent of batching; models that do not fit a T4 (15 GB)
# in float32 load in 4-bit with float16 compute. Report the dtype of each model in the paper.
MODELS = [
    {"name": "HuggingFaceTB/SmolLM2-1.7B-Instruct", "dtype": "float32", "load_in_4bit": False, "batch_size": 8},
    {"name": "Qwen/Qwen2.5-1.5B-Instruct", "dtype": "float32", "load_in_4bit": False, "batch_size": 8},
    {"name": "microsoft/Phi-3.5-mini-instruct", "dtype": "float16", "load_in_4bit": True, "batch_size": 8},
    {"name": "Qwen/Qwen2.5-7B-Instruct", "dtype": "float16", "load_in_4bit": True, "batch_size": 8},
]
RUN_DIRECT = False  # the direct run is complete; True only to redo it
SMOKE_TEST = True
SMOKE_ITEMS = 24


def result_path(model_name):
    suffix = "-smoke" if SMOKE_TEST else ""
    return OUT_DIR / f"logprobs-{model_name.split('/')[-1]}{suffix}.jsonl"

## Batch-invariance check (smoke test only)

P(Yes) for an item should not depend on which other items share its batch. Values above about 0.01 mean the precision setting for that model should change.

In [ ]:
from causal_evaluation.models.logprob import check_batch_invariance

if RUN_DIRECT and SMOKE_TEST:
    probe = [items[0], items[1500], items[2999], items[7]]
    for spec in MODELS:
        diff = check_batch_invariance(spec["name"], probe, spec["load_in_4bit"], spec["dtype"])
        print(f"{spec['name']}: max |batched - alone| = {diff:.4f}" + ("  <-- check dtype" if diff > 0.01 else ""))

## Run

In [ ]:
import gc

import torch

from causal_evaluation.models.logprob import run_model

run_items = items[:SMOKE_ITEMS] if SMOKE_TEST else items
for spec in MODELS if RUN_DIRECT else []:
    path = result_path(spec["name"])
    scored = run_model(
        run_items,
        spec["name"],
        path,
        batch_size=spec["batch_size"],
        load_in_4bit=spec["load_in_4bit"],
        dtype=spec["dtype"],
    )
    print(f"{spec['name']}: {scored} new items -> {path.name}")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Diagnostics

`answer_mass` is the probability the model puts on Yes/No tokens at all. If its median is low (say below 0.5) the model wants to start with something else (look at `top_token`) and P(Yes) is less meaningful for that model.

In [ ]:
import pandas as pd

from causal_evaluation.evaluation.variant_analysis import load_scored, rate_table

paths = [
    p
    for p in sorted(OUT_DIR.glob("logprobs-*.jsonl"))
    if p.name.endswith("-smoke.jsonl") == SMOKE_TEST and "causal_cot" not in p.name
]
scored = load_scored(OUT_DIR / "variants-sample.jsonl", paths)
display(scored.groupby("model").agg(items=("item_id", "size"), median_answer_mass=("answer_mass", "median"), yes_rate=("pred", lambda s: (s == "yes").mean())))
display(scored.groupby("model")["top_token"].agg(lambda s: s.value_counts().head(3).to_dict()))
rate_table(scored, ["model", "variant"])

## CausalCoT run

Same items and the same Yes/No scoring, but the model first writes the six CausalCoT steps of Jin et al. (2023): causal graph, query type, formal expression, available data, estimand, calculation. Its reasoning stays in the conversation, a second turn asks *"Based on all the reasoning above, answer the initial question with just Yes or No."*, and P(Yes) is read from the first answer token.

Generation is much slower than the direct run, so this uses a balanced subset: the first `COT_PAIRS` matched pairs and the first `COT_NONSENSE` nonsense items (both already spread evenly over rung and answer). Start with `COT_SMOKE = True` to read one reasoning trace and get a time estimate, then set it to `False`.

In [ ]:
from collections import Counter

COT_MODELS = [spec["name"] for spec in MODELS]  # the four models, run in the cells below
COT_PAIRS = 150
COT_NONSENSE = 150
COT_MAX_NEW_TOKENS = 1024  # first run used 512; raising it redoes only truncated traces
COT_BATCH_SIZE = 8
COT_SMOKE = False  # True for a quick 8-item check
COT_SMOKE_ITEMS = 8

specs = {spec["name"]: spec for spec in MODELS}
chosen_pairs = set(sorted({i["pair_id"] for i in items if i["pair_id"]})[:COT_PAIRS])
nonsense_items = [i for i in items if i["variant"] == "noncommonsense"][:COT_NONSENSE]
cot_items = [i for i in items if i["pair_id"] in chosen_pairs] + nonsense_items
print(f"{len(cot_items)} items: {len(chosen_pairs)} pairs + {len(nonsense_items)} nonsense")
print(Counter((i["variant"], i["rung"], i["answer"]) for i in cot_items))


def cot_path(model_name):
    suffix = "-smoke" if COT_SMOKE else ""
    return OUT_DIR / f"logprobs-{model_name.split('/')[-1]}-causal_cot{suffix}.jsonl"

In [ ]:
import time


def run_cot_model(name):
    """Run CausalCoT for one model. Safe to rerun: it skips items already saved."""
    spec = specs[name]
    run_items = cot_items[:COT_SMOKE_ITEMS] if COT_SMOKE else cot_items
    start = time.time()
    scored_now = run_model(
        run_items,
        name,
        cot_path(name),
        batch_size=COT_BATCH_SIZE,
        load_in_4bit=spec["load_in_4bit"],
        dtype=spec["dtype"],
        prompt_strategy="causal_cot",
        max_new_tokens=COT_MAX_NEW_TOKENS,
    )
    minutes = (time.time() - start) / 60
    print(f"{name}: {scored_now} new items in {minutes:.1f} min -> {cot_path(name).name}")
    if COT_SMOKE and scored_now:
        hours = minutes / scored_now * len(cot_items) / 60
        print(f"  rough estimate for all {len(cot_items)} items: {hours:.1f} h (includes loading)")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

### Items pending per model

With a higher `COT_MAX_NEW_TOKENS` than a previous run, only the traces that were **truncated** are redone. Greedy decoding gives the same reasoning for items that already finished, so this is equivalent to running everything at the new limit. The new rows are appended to the same file, and the analysis keeps the latest row per item.

In [ ]:
from causal_evaluation.models.logprob import done_item_ids

run_ids = {i["item_id"] for i in (cot_items[:COT_SMOKE_ITEMS] if COT_SMOKE else cot_items)}
for name in COT_MODELS:
    done = done_item_ids(cot_path(name), name, "causal_cot", COT_MAX_NEW_TOKENS)
    print(f"{name}: {len(run_ids - done)} items to run (limit {COT_MAX_NEW_TOKENS} tokens)")

### One cell per model

If a cell fails or Colab disconnects, run the cells above with **Runtime → Run before** (Ctrl+F8 on the cell), then rerun only that model's cell. It continues from the results saved in Drive.

In [ ]:
run_cot_model("HuggingFaceTB/SmolLM2-1.7B-Instruct")

In [ ]:
run_cot_model("Qwen/Qwen2.5-1.5B-Instruct")

In [ ]:
run_cot_model("microsoft/Phi-3.5-mini-instruct")

In [ ]:
run_cot_model("Qwen/Qwen2.5-7B-Instruct")

### CausalCoT diagnostics

- `truncated`: share of reasoning traces cut off at `COT_MAX_NEW_TOKENS`. If it is still high (say above 20%), raise the limit again (e.g. 1536) and rerun the model cells: only the traces still truncated are redone.
- `median_answer_mass`: should stay close to 1 (the model answers Yes/No after reasoning).
- Read at least one trace: the reasoning should use the graph and numbers from the prompt.

Results are written to `logprobs-<model>-causal_cot.jsonl`. Copy them to `data/interim/` with the others; notebook 03 labels them `<model>-causal_cot`.

In [ ]:
cot_paths = [
    p
    for p in sorted(OUT_DIR.glob("logprobs-*-causal_cot*.jsonl"))
    if p.name.endswith("-smoke.jsonl") == COT_SMOKE
]
cot = load_scored(OUT_DIR / "variants-sample.jsonl", cot_paths)
display(
    cot.groupby("model").agg(
        items=("item_id", "size"),
        truncated=("reasoning_truncated", "mean"),
        median_answer_mass=("answer_mass", "median"),
        yes_rate=("pred", lambda s: (s == "yes").mean()),
    )
)
example = cot.iloc[0]
print(f"{example['model']} | {example['item_id']} | correct answer: {example['answer']} | P(Yes) = {example['p_yes']:.2f}\n")
print(example["reasoning"][:3000])

In [ ]:
rate_table(cot, ["model", "variant"])